# Cross-Model Comparison Analysis

This notebook loads the results from `scripts/run_model_comparison.py` and produces:
1. Comparison tables across all models
2. Bar charts of key metrics (BLEU, BERTScore, Judge)
3. Robustness recovery ratio heatmaps by model and noise type
4. Statistical significance tests between models

**Prerequisites:** Run `python scripts/run_model_comparison.py` first to generate results.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["figure.dpi"] = 150

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
COMPARISON_DIR = PROJECT_ROOT / "data" / "outputs" / "comparison"

print(f"Looking for results in: {COMPARISON_DIR}")
print(f"Files found: {list(COMPARISON_DIR.glob('*.json')) if COMPARISON_DIR.exists() else 'Directory not found'}")

## 1. Load Results

In [ ]:
# Load the full comparison results
full_path = COMPARISON_DIR / "model_comparison_full.json"
if not full_path.exists():
    raise FileNotFoundError(
        f"{full_path} not found. Run `python scripts/run_model_comparison.py` first."
    )

with open(full_path, "r", encoding="utf-8") as f:
    all_results = json.load(f)

# Filter out failed runs
results = [r for r in all_results if r.get("model") != "FAILED"]
failed = [r for r in all_results if r.get("model") == "FAILED"]

print(f"Loaded {len(results)} successful model runs, {len(failed)} failed.")
if failed:
    for f in failed:
        print(f"  FAILED: {f['config']} — {f.get('error', 'unknown')}")

# Also load the CSV summary
comparison_csv = COMPARISON_DIR / "model_comparison.csv"
if comparison_csv.exists():
    comparison_df = pd.read_csv(comparison_csv)
    display(comparison_df)

## 2. Overall Pipeline Comparison (Bar Charts)

In [ ]:
# Build a tidy dataframe for plotting
rows = []
for r in results:
    label = r["label"]
    for metric in ["bleu", "bertscore", "judge"]:
        for pipeline in ["clean", "noisy", "repaired"]:
            rows.append({
                "Model": label,
                "Metric": metric.upper() if metric != "bertscore" else "BERTScore",
                "Pipeline": pipeline.capitalize(),
                "Score": r[f"{metric}_{pipeline}_mean"],
            })

plot_df = pd.DataFrame(rows)

for metric_name in plot_df["Metric"].unique():
    subset = plot_df[plot_df["Metric"] == metric_name]
    fig, ax = plt.subplots(figsize=(max(10, len(results) * 2), 6))
    sns.barplot(data=subset, x="Model", y="Score", hue="Pipeline",
                palette=["#2ecc71", "#e74c3c", "#3498db"], ax=ax)
    ax.set_title(f"{metric_name} by Model and Pipeline")
    ax.set_ylabel(metric_name)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(COMPARISON_DIR / f"chart_{metric_name.lower()}_by_model.png", dpi=150)
    plt.show()

## 3. Recovery Ratio Comparison

In [ ]:
# Recovery ratio: what fraction of noise degradation does repair recover?
recovery_rows = []
for r in results:
    recovery_rows.append({
        "Model": r["label"],
        "BLEU Recovery %": r["bleu_recovery_ratio"] * 100,
        "BERTScore Recovery %": r["bertscore_recovery_ratio"] * 100,
        "Judge Recovery %": r["judge_recovery_ratio"] * 100,
    })

recovery_df = pd.DataFrame(recovery_rows).set_index("Model")

fig, ax = plt.subplots(figsize=(max(10, len(results) * 1.5), 6))
recovery_df.plot(kind="bar", ax=ax, color=["#2ecc71", "#3498db", "#9b59b6"])
ax.set_title("Recovery Ratio by Model (% of degradation recovered)")
ax.set_ylabel("Recovery Ratio (%)")
ax.axhline(y=100, color="gray", linestyle="--", alpha=0.5, label="Full recovery")
ax.axhline(y=0, color="red", linestyle="--", alpha=0.3)
plt.xticks(rotation=30, ha="right")
plt.legend(loc="best")
plt.tight_layout()
plt.savefig(COMPARISON_DIR / "chart_recovery_ratio.png", dpi=150)
plt.show()

display(recovery_df.round(1))

## 4. Per Noise-Type Heatmap

In [ ]:
# Heatmap: BERTScore recovery by model x noise type
heatmap_rows = []
for r in results:
    label = r["label"]
    for nt_data in r.get("per_noise_type", []):
        clean = nt_data["bertscore_clean_mean"]
        noisy = nt_data["bertscore_noisy_mean"]
        repaired = nt_data["bertscore_repaired_mean"]
        degradation = clean - noisy
        recovery = repaired - noisy
        ratio = (recovery / degradation * 100) if degradation != 0 else 0
        heatmap_rows.append({
            "Model": label,
            "Noise Type": nt_data["noise_type"],
            "Recovery %": ratio,
        })

if heatmap_rows:
    heatmap_df = pd.DataFrame(heatmap_rows).pivot(index="Model", columns="Noise Type", values="Recovery %")

    fig, ax = plt.subplots(figsize=(12, max(4, len(results) * 0.8)))
    sns.heatmap(heatmap_df, annot=True, fmt=".0f", cmap="RdYlGn", center=50,
                vmin=0, vmax=100, ax=ax, linewidths=0.5)
    ax.set_title("BERTScore Recovery (%) by Model and Noise Type")
    plt.tight_layout()
    plt.savefig(COMPARISON_DIR / "chart_heatmap_recovery.png", dpi=150)
    plt.show()
else:
    print("No per-noise-type data available.")

## 5. Runtime Comparison

In [ ]:
runtime_rows = []
for r in results:
    if "runtime_seconds" in r:
        runtime_rows.append({
            "Model": r["label"],
            "Runtime (min)": r["runtime_seconds"] / 60,
            "Samples": r["n_examples"],
            "Sec/Sample": r["runtime_seconds"] / r["n_examples"] if r["n_examples"] > 0 else 0,
        })

if runtime_rows:
    runtime_df = pd.DataFrame(runtime_rows)
    fig, ax = plt.subplots(figsize=(max(8, len(results) * 1.5), 5))
    sns.barplot(data=runtime_df, x="Model", y="Runtime (min)", color="#3498db", ax=ax)
    ax.set_title("Total Runtime by Model")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(COMPARISON_DIR / "chart_runtime.png", dpi=150)
    plt.show()
    display(runtime_df.round(2))

## 6. Summary Table (Publication-Ready)

In [ ]:
summary_rows = []
for r in results:
    summary_rows.append({
        "Model": r["label"],
        "BLEU (C/N/R)": f"{r['bleu_clean_mean']:.1f} / {r['bleu_noisy_mean']:.1f} / {r['bleu_repaired_mean']:.1f}",
        "BERTScore (C/N/R)": f"{r['bertscore_clean_mean']:.3f} / {r['bertscore_noisy_mean']:.3f} / {r['bertscore_repaired_mean']:.3f}",
        "Judge (C/N/R)": f"{r['judge_clean_mean']:.2f} / {r['judge_noisy_mean']:.2f} / {r['judge_repaired_mean']:.2f}",
        "BLEU Recov.": f"{r['bleu_recovery_ratio']:.0%}" if not np.isnan(r['bleu_recovery_ratio']) else "N/A",
        "BERT Recov.": f"{r['bertscore_recovery_ratio']:.0%}" if not np.isnan(r['bertscore_recovery_ratio']) else "N/A",
        "Judge Recov.": f"{r['judge_recovery_ratio']:.0%}" if not np.isnan(r['judge_recovery_ratio']) else "N/A",
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

# Save as CSV and LaTeX
summary_df.to_csv(COMPARISON_DIR / "summary_table.csv", index=False)
print("\nLaTeX table:")
print(summary_df.to_latex(index=False, escape=False))